# 🧠 Bloque 2 – Modelos Básicos con Scikit-learn

Este notebook te permitirá practicar clasificación y regresión con `scikit-learn`, incluyendo métricas de evaluación y visualización de resultados.

Usaremos dos datasets:
- **Titanic**: clasificación (sobrevivir o no)
- **Ames Housing**: regresión (predecir precio de vivienda)

> **Antes de ejecutarlo:** es un cuaderno de repaso y no el primer recorrido
> de clase. Titanic carga datos con Seaborn y prepara después de separar;
> necesita la caché o acceso al dataset. Ames usa
> OpenML. La sección de GridSearch siguiente sí es independiente y trabaja
> con otro conjunto incorporado, sin reutilizar los datos de regresión.


## ⚙️ Setup inicial

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

## 🔹 Clasificación – Titanic

Separamos primero las características y la respuesta; después reservamos test.
El preprocesado vive dentro de un `Pipeline`: cuando llamamos a `fit` aprende
medianas, categorías y modelo **solo con entrenamiento**. Al llamar a `predict`,
aplica esas mismas transformaciones al conjunto de prueba sin volver a ajustarlas.

¿Por qué? Si el codificador descubre categorías en todo el conjunto antes de
separar, utiliza información de los datos que deberían simular casos nuevos.
Esa fuga no implica que todos los resultados previos sean necesariamente falsos,
pero debilita la independencia de la evaluación. El mismo problema afecta a
medianas y otras estadísticas aprendidas de todos los datos.

`handle_unknown="ignore"` permite procesar categorías no vistas en entrenamiento
sin aprenderlas del test; se representan con ceros en ese grupo de columnas.
Aquí cargamos el Titanic de Seaborn, cuya disponibilidad depende de su caché o
de la red. Para el recorrido autónomo de clase, usa
`03-practicas/01-primer-clasificador-titanic.ipynb` y su CSV local.

In [ ]:
# Primero seleccionamos X e y y dividimos; no aprendemos categorías ni medianas.
df = sns.load_dataset("titanic")
columnas_numericas = ["age", "fare"]
columnas_categoricas = ["pclass", "sex", "embarked"]
X_titanic = df[columnas_numericas + columnas_categoricas].copy()
y_titanic = df["survived"].copy()
X_titanic_train, X_titanic_test, y_titanic_train, y_titanic_test = train_test_split(
    X_titanic, y_titanic, test_size=0.2, random_state=42, stratify=y_titanic
)
assert "survived" not in X_titanic.columns
assert set(X_titanic_train.index).isdisjoint(X_titanic_test.index)
print("Entrenamiento:", X_titanic_train.shape, "| prueba:", X_titanic_test.shape)


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

preparacion_titanic = ColumnTransformer([
    ("numericas", SimpleImputer(strategy="median"), columnas_numericas),
    ("categoricas", Pipeline([
        ("imputar", SimpleImputer(strategy="most_frequent")),
        ("codificar", OneHotEncoder(handle_unknown="ignore")),
    ]), columnas_categoricas),
])
clf_titanic = Pipeline([
    ("preparar", preparacion_titanic),
    ("modelo", RandomForestClassifier(random_state=42)),
])
# fit aprende preparación y modelo únicamente a partir de train.
clf_titanic.fit(X_titanic_train, y_titanic_train)
# predict transforma test con lo aprendido, sin volver a ajustar.
pred_titanic = clf_titanic.predict(X_titanic_test)

print("Accuracy:", accuracy_score(y_titanic_test, pred_titanic))
print(classification_report(y_titanic_test, pred_titanic, zero_division=0))


In [ ]:
sns.heatmap(confusion_matrix(y_titanic_test, pred_titanic), annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.title("Matriz de confusión")
plt.show()


## 🔹 Regresión – Ames Housing (versión ligera)

In [ ]:
from sklearn.datasets import fetch_openml

ames = fetch_openml(name="house_prices", as_frame=True)
df = ames.frame[['SalePrice', 'GrLivArea', 'OverallQual', 'YearBuilt', 'GarageCars', 'FullBath']].dropna()

X = df.drop('SalePrice', axis=1)
y = df['SalePrice']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LinearRegression

reg = LinearRegression()
reg.fit(X_train_scaled, y_train)
y_pred = reg.predict(X_test_scaled)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2:", r2_score(y_test, y_pred))

In [ ]:
plt.scatter(y_test, y_pred, alpha=0.3)
plt.xlabel("Real")
plt.ylabel("Predicción")
plt.title("Regresión: Valor real vs predicho")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.show()

## 🔄 GridSearch básico — después del primer recorrido

Ejemplo independiente con el conjunto `wine` incluido en scikit-learn:
clasificamos tres variedades, no precios de Ames ni supervivencia. Reservamos
test una sola vez; dentro de entrenamiento, validación cruzada estratificada
elige entre configuraciones. El imputador va dentro del `Pipeline`, de manera
que se ajusta de nuevo en cada pliegue de entrenamiento.

`best_score_` es la media de validación cruzada, no la exactitud de test.
El estimador seleccionado se reajusta con todo el entrenamiento. Luego
evaluamos una sola vez el modelo fijado en test; no lo usamos para seguir
eligiendo hiperparámetros. Es una ampliación, no parte de la primera sesión.

In [ ]:
# Independiente de Titanic/Ames y de sus variables X_train/y_train.
from sklearn.datasets import load_wine
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline

vino = load_wine(as_frame=True)
X_vino, y_vino = vino.data, vino.target
X_vino_train, X_vino_test, y_vino_train, y_vino_test = train_test_split(
    X_vino, y_vino, test_size=0.2, stratify=y_vino, random_state=42
)
assert set(X_vino_train.index).isdisjoint(X_vino_test.index)
pipeline_vino = Pipeline([
    ("imputar", SimpleImputer(strategy="median")),
    ("clasificador", RandomForestClassifier(random_state=42)),
])
param_grid_vino = {
    "clasificador__n_estimators": [50, 100],
    "clasificador__max_depth": [3, 5, None],
}
cv_vino = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
busqueda_vino = GridSearchCV(
    pipeline_vino, param_grid_vino, cv=cv_vino, scoring="accuracy", refit=True
)
busqueda_vino.fit(X_vino_train, y_vino_train)
print("Mejor combinación:", busqueda_vino.best_params_)
print("Exactitud media en validación cruzada:", round(busqueda_vino.best_score_, 3))
pred_vino = busqueda_vino.predict(X_vino_test)
print("Exactitud final en test reservado:", round(accuracy_score(y_vino_test, pred_vino), 3))
